<a href="https://colab.research.google.com/github/khalilosss11/f1_Bahrain_GP_2026_predictor/blob/main/f1_Bahrain_GP_2026_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastf1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.7/424.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.2.3
    Uninstalling msgpack-1.2.3:
      Successfully uninstalled msgpack-1.2.3


In [2]:
# Reproduces the Bahrain GP (held at Sepang) prediction as it was before the race
CUTOFF_DATE = '2026-10-04'                  # race day: only data strictly BEFORE this date is used
CIRCUIT_HISTORY_NAME = 'Malaysian Grand Prix'   # Sepang history comes from Kaggle's old name
SNAPSHOT = 'fastf1_2025_2026_snapshot.csv'

In [3]:
# Loads the 1950-2024 F1 historical dataset from Kaggle (core training data)
import pandas as pd
import numpy as np

from google.colab import userdata
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET = "rohanrao/formula-1-world-championship-1950-2020"
def load(file_path):
    return kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, file_path)

races = load("races.csv")
results = load("results.csv")
drivers = load("drivers.csv")
constructors = load("constructors.csv")
qualifying = load("qualifying.csv")
driver_standings = load("driver_standings.csv")

/tmp/ipykernel_2448/3846161176.py:14: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, file_path)


100%|██████████| 160k/160k [00:00<00:00, 2.10MB/s]
/tmp/ipykernel_2448/3846161176.py:14: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, file_path)


Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.


/tmp/ipykernel_2448/3846161176.py:14: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, file_path)


Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.


...

Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.


/tmp/ipykernel_2448/3846161176.py:14: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, file_path)


Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.


/tmp/ipykernel_2448/3846161176.py:14: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  return kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, file_path)


Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.


In [4]:
# Merge Kaggle tables into one clean table
# Joins races + results + drivers + constructors + qualifying into one row
# per driver per race, and fixes data types (position, date, '\N' -> NaN)
df = results.merge(races[['raceId','year','round','circuitId','date','name']], on='raceId', how='left')
df = df.merge(drivers[['driverId','driverRef','code','forename','surname']], on='driverId', how='left')
df = df.merge(constructors[['constructorId','constructorRef','name']].rename(columns={'name':'constructor_name'}), on='constructorId', how='left')
q = qualifying[['raceId','driverId','position']].rename(columns={'position':'quali_position'})
df = df.merge(q, on=['raceId','driverId'], how='left')

df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date','raceId']).reset_index(drop=True)
df['won'] = (df['position']==1).astype(int)

print(df.shape)

(26759, 31)


In [5]:
# points FastF1 cache at Google Drive, so fetched data
# survives Colab restarts instead of being lost every time
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/f1_project/fastf1_cache', exist_ok=True)

import fastf1
fastf1.Cache.enable_cache('/content/drive/MyDrive/f1_project/fastf1_cache')

Mounted at /content/drive


In [6]:
# Defines a reusable function that pulls every completed race + qualifying
# session for a given season, skipping any race that hasn't happened yet
def fetch_season(season):
    rows = []
    schedule = fastf1.get_event_schedule(season, include_testing=False)
    for _, event in schedule.iterrows():
        round_num, event_name = event['RoundNumber'], event['EventName']
        try:
            race = fastf1.get_session(season, round_num, 'R')
            race.load(laps=False, telemetry=False, weather=False, messages=False)
        except Exception as e:
            print(f"skip {season} R{round_num} {event_name}: {e}")
            continue
        if race.results is None or race.results.empty:
            continue
        try:
            quali = fastf1.get_session(season, round_num, 'Q')
            quali.load(laps=False, telemetry=False, weather=False, messages=False)
            quali_pos = quali.results.set_index('Abbreviation')['Position'].to_dict()
        except Exception:
            quali_pos = {}
        for _, r in race.results.iterrows():
            code = r['Abbreviation']
            rows.append({
                'year': season, 'round': round_num, 'name': event_name,
                'date': event['EventDate'], 'code': code,
                'constructor_name': r['TeamName'], 'grid': r['GridPosition'],
                'position': r['Position'], 'quali_position': quali_pos.get(code),
                'points': r['Points'],
            })
    return pd.DataFrame(rows)

In [7]:
# API and can be rate-limited (500 calls/hour), so only re-run if needed
# First run: fetch from FastF1 and save a snapshot. Later runs read the file instead.
# The snapshot holds ALL fetched races; the cutoff below is what removes the future ones.
import os
if os.path.exists(SNAPSHOT):
    new_rows = pd.read_csv(SNAPSHOT)
else:
    new_rows = pd.concat([fetch_season(2025), fetch_season(2026)], ignore_index=True)
    new_rows.to_csv(SNAPSHOT, index=False)

# Drop every race on or after race day, so the prediction can't see its own result
new_rows['date'] = pd.to_datetime(new_rows['date'])
new_rows = new_rows[new_rows['date'] < pd.Timestamp(CUTOFF_DATE)].copy()

print(new_rows.shape)
print("Last race used:", new_rows.sort_values('date').iloc[-1][['name', 'date']].to_dict())
assert new_rows['date'].max() < pd.Timestamp(CUTOFF_DATE)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
           

(809, 10)
Last race used: {'name': 'Azerbaijan Grand Prix', 'date': Timestamp('2026-09-26 00:00:00')}


In [8]:
# Stacks historical (Kaggle) and live (FastF1) data together using shared
# text-based keys (code, constructor_name, name) so driver history flows
# continuously from 2024 into 2025/2026 instead of resetting
base_cols = ['year','round','name','date','code','constructor_name','grid','position','quali_position','points']
df_combined = pd.concat([df[base_cols], new_rows[base_cols]], ignore_index=True)

df_combined['date'] = pd.to_datetime(df_combined['date'])
df_combined['position'] = pd.to_numeric(df_combined['position'], errors='coerce')
df_combined = df_combined.sort_values(['date']).reset_index(drop=True)
df_combined['won'] = (df_combined['position']==1).astype(int)

print(df_combined.shape)

(27568, 11)


In [9]:
# Builds the actual signals the model learns from: recent form (last 5
# races), circuit-specific history, and championship standing - all
# computed with shift(1) so a race never "sees" its own result (no leakage)
WINDOW = 5

df_combined['driver_recent_form'] = df_combined.groupby('code')['position'].transform(
    lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
df_combined['constructor_recent_form'] = df_combined.groupby('constructor_name')['position'].transform(
    lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
df_combined['driver_circuit_history'] = df_combined.groupby(['code','name'])['position'].transform(
    lambda s: s.shift(1).expanding().mean())
df_combined['constructor_circuit_history'] = df_combined.groupby(['constructor_name','name'])['position'].transform(
    lambda s: s.shift(1).expanding().mean())
df_combined['champ_points_entering'] = df_combined.groupby(['year','code'])['points'].transform(
    lambda s: s.shift(1).cumsum())
df_combined['champ_position_entering'] = df_combined.groupby('year')['champ_points_entering'].rank(
    ascending=False, method='min')

df_combined.tail()

,year,round,name,date,code,constructor_name,grid,position,quali_position,points,won,driver_recent_form,constructor_recent_form,driver_circuit_history,constructor_circuit_history,champ_points_entering,champ_position_entering
27563,2026,15,Azerbaijan Grand Prix,2026-09-26,VER,Red Bull,8.0,2.0,8.0,18.0,0,6.4,5.6,2.600000,3.666667,133.0,22.0
27564,2026,15,Azerbaijan Grand Prix,2026-09-26,RUS,Mercedes,1.0,1.0,1.0,25.0,1,7.8,2.8,6.200000,5.125000,177.0,6.0
27565,2026,15,Azerbaijan Grand Prix,2026-09-26,ALO,Aston Martin,21.0,21.0,19.0,0.0,0,16.0,17.6,7.714286,10.222222,3.0,201.0
27566,2026,15,Azerbaijan Grand Prix,2026-09-26,SAI,Williams,14.0,10.0,9.0,1.0,0,16.4,16.8,7.714286,11.714286,6.0,168.0
27567,2026,15,Azerbaijan Grand Prix,2026-09-26,STR,Aston Martin,22.0,22.0,21.0,0.0,0,18.8,20.0,11.285714,11.300000,0.0,239.0


In [10]:
# Selects just the columns the model needs and drops rows with no grid
# position at all (can't train on those)
feature_cols_combined = [
    'year', 'round', 'name', 'date', 'code', 'constructor_name',
    'grid', 'quali_position',
    'driver_recent_form', 'constructor_recent_form',
    'driver_circuit_history', 'constructor_circuit_history',
    'champ_points_entering', 'champ_position_entering',
    'position', 'won',
]

training_table_combined = df_combined[feature_cols_combined].dropna(subset=['grid']).copy()
print(training_table_combined.shape)

(27568, 16)


In [11]:
# Holds out 2022-2024 as an honest test set (model never trains on these),
# trains an XGBoost classifier, wrapped in calibration for realistic
# probabilities, with scale_pos_weight to handle the ~19:1 class imbalance
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV

FEATURES = [
    'grid', 'quali_position',
    'driver_recent_form', 'constructor_recent_form',
    'driver_circuit_history', 'constructor_circuit_history',
    'champ_points_entering', 'champ_position_entering',
]

TEST_SEASONS = [2022, 2023, 2024]

train_df2 = training_table_combined[~training_table_combined['year'].isin(TEST_SEASONS)]
test_df2 = training_table_combined[training_table_combined['year'].isin(TEST_SEASONS)]

train_clean2 = train_df2.dropna(subset=FEATURES + ['won'])
test_clean2 = test_df2.dropna(subset=FEATURES + ['won'])

print(f"Train: {len(train_clean2)} rows")
print(f"Test:  {len(test_clean2)} rows")

X_train2, y_train2 = train_clean2[FEATURES], train_clean2['won']
X_test2, y_test2 = test_clean2[FEATURES], test_clean2['won']

base_model2 = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train2 == 0).sum() / (y_train2 == 1).sum(),
    eval_metric='logloss', random_state=42,
)
model2 = CalibratedClassifierCV(base_model2, method='sigmoid', cv=5)
model2.fit(X_train2, y_train2)
print("Model trained on combined data.")

Train: 6348 rows
Test:  1047 rows
Model trained on combined data.


In [12]:
# Fit a plain (uncalibrated) version just to inspect feature importances
base_model2.fit(X_train2, y_train2)

importances = base_model2.feature_importances_
for feat, imp in sorted(zip(FEATURES, importances), key=lambda x: -x[1]):
    print(f"{feat}: {imp:.3f}")

grid: 0.451
quali_position: 0.257
driver_recent_form: 0.080
driver_circuit_history: 0.048
constructor_recent_form: 0.047
champ_position_entering: 0.043
constructor_circuit_history: 0.040
champ_points_entering: 0.034


In [13]:
# Checks how often the model's top pick actually won, on races it never
# trained on - this is the real measure of whether the model works
from sklearn.metrics import log_loss

def top1_accuracy(df, prob_col='win_prob'):
    correct, total = 0, 0
    for race_id, g in df.groupby(['year', 'round']):
        total += 1
        correct += int(g.loc[g[prob_col].idxmax(), 'won'] == 1)
    return correct / total if total else float('nan')

test_clean2 = test_clean2.copy()
test_clean2['win_prob_raw'] = model2.predict_proba(X_test2)[:, 1]
test_clean2['win_prob'] = test_clean2.groupby(['year','round'])['win_prob_raw'].transform(lambda s: s / s.sum())

acc2 = top1_accuracy(test_clean2)
ll2 = log_loss(test_clean2['won'], test_clean2['win_prob'].clip(1e-6, 1-1e-6))
print(f"Top-1 accuracy: {acc2:.3f}, Log loss: {ll2:.4f}")

Top-1 accuracy: 0.524, Log loss: 0.1312


In [14]:
# Gathers the most recent 2026 race's drivers/teams, their rolling recent
# form, and their historical Sepang performance (circuit hasn't hosted a
# round since 2017, so this comes from Kaggle's "Malaysian Grand Prix" rows
rows_2026 = df_combined[df_combined['year']==2026]
latest_round = rows_2026['round'].max()
grid = rows_2026[rows_2026['round']==latest_round][['code','constructor_name']].drop_duplicates().reset_index(drop=True)

recent = df_combined.sort_values('date')
driver_form = recent.groupby('code')['position'].apply(lambda s: s.tail(5).mean()).rename('driver_recent_form')
team_form = recent.groupby('constructor_name')['position'].apply(lambda s: s.tail(5).mean()).rename('constructor_recent_form')
driver_avg_grid = recent.groupby('code')['grid'].apply(lambda s: s.tail(5).mean()).rename('avg_grid')

sepang = df[df['name']=='Malaysian Grand Prix']
driver_sepang = sepang.groupby('code')['position'].mean().rename('driver_circuit_history')
team_sepang = sepang.groupby('constructor_name')['position'].mean().rename('constructor_circuit_history')

champ = rows_2026.groupby('code')['points'].sum().rename('champ_points_entering')

pred = grid.merge(driver_form, on='code', how='left') \
           .merge(team_form, on='constructor_name', how='left') \
           .merge(driver_avg_grid, on='code', how='left') \
           .merge(driver_sepang, on='code', how='left') \
           .merge(team_sepang, on='constructor_name', how='left') \
           .merge(champ, on='code', how='left')

pred['driver_circuit_history'] = pred['driver_circuit_history'].fillna(pred['constructor_circuit_history'])
pred['driver_circuit_history'] = pred['driver_circuit_history'].fillna(pred['driver_circuit_history'].median())
pred['constructor_circuit_history'] = pred['constructor_circuit_history'].fillna(pred['driver_circuit_history'])

pred['grid'] = pred['avg_grid']
pred['quali_position'] = pred['avg_grid']
pred['champ_position_entering'] = pred['champ_points_entering'].rank(ascending=False, method='min')

for col in FEATURES:
    pred[col] = pred[col].fillna(pred[col].median())

In [16]:
# Real Oct 3 2026 qualifying results (grid includes Hadjar's 5-place
# penalty, which promotes Antonelli to P3 on the actual starting grid)
quali_results = pd.DataFrame([
    {'code': 'VER', 'quali_position': 1, 'grid': 1},
    {'code': 'HAM', 'quali_position': 2, 'grid': 2},
    {'code': 'ANT', 'quali_position': 4, 'grid': 3},   # promoted due to Hadjar penalty
    {'code': 'HAD', 'quali_position': 3, 'grid': 8},   # 5-place grid penalty applied
    {'code': 'LEC', 'quali_position': 5, 'grid': 4},
    {'code': 'NOR', 'quali_position': 6, 'grid': 5},
    {'code': 'PIA', 'quali_position': 7, 'grid': 6},
    {'code': 'RUS', 'quali_position': 8, 'grid': 7},
    {'code': 'GAS', 'quali_position': 9, 'grid': 9},
    {'code': 'BOR', 'quali_position': 10, 'grid': 10},
    {'code': 'LAW', 'quali_position': 11, 'grid': 11},
    {'code': 'ALO', 'quali_position': 12, 'grid': 12},
    {'code': 'SAI', 'quali_position': 13, 'grid': 13},
    {'code': 'STR', 'quali_position': 14, 'grid': 14},
    {'code': 'COL', 'quali_position': 15, 'grid': 15},
    {'code': 'LIN', 'quali_position': 16, 'grid': 16},
    {'code': 'HUL', 'quali_position': 17, 'grid': 17},
    {'code': 'BEA', 'quali_position': 18, 'grid': 18},
    {'code': 'OCO', 'quali_position': 19, 'grid': 19},
    {'code': 'ALB', 'quali_position': 20, 'grid': 20},
    {'code': 'BOT', 'quali_position': 21, 'grid': 21},
    {'code': 'PER', 'quali_position': 22, 'grid': 22},
])
quali_results

,code,quali_position,grid
0,VER,1,1
1,HAM,2,2
2,ANT,4,3
3,HAD,3,8
4,LEC,5,4
5,NOR,6,5
6,PIA,7,6
7,RUS,8,7
8,GAS,9,9
9,BOR,10,10


In [17]:
print(pred.shape)
print(grid.shape)
pred.head()

(22, 11)
(22, 2)


,code,constructor_name,driver_recent_form,constructor_recent_form,avg_grid,driver_circuit_history,constructor_circuit_history,champ_points_entering,grid,quali_position,champ_position_entering
0,PIA,McLaren,10.4,9.6,4.6,6.068966,6.068966,99.0,4.6,4.6,7.0
1,PER,Cadillac F1 Team,17.6,17.4,19.8,7.200000,7.200000,0.0,19.8,19.8,20.0
2,BOR,Audi,12.6,12.2,12.4,6.542857,6.542857,10.0,12.4,12.4,14.0
3,ALB,Williams,17.2,15.4,16.6,12.500000,8.384615,5.0,16.6,16.6,18.0
4,COL,Alpine F1 Team,12.4,12.2,10.4,6.542857,6.542857,27.0,10.4,10.4,12.0


In [18]:
# Runs the trained model on the real 2026 grid and normalizes win
# probabilities across the field so they sum to 100%
pred = pred.drop(columns=['grid','quali_position'])
pred = pred.merge(quali_results, on='code', how='left')

raw = model2.predict_proba(pred[FEATURES])[:, 1]
pred['win_probability'] = (raw / raw.sum() * 100).round(1)
result = pred[['code','constructor_name','win_probability']].sort_values('win_probability', ascending=False)
result

,code,constructor_name,win_probability
17,VER,Red Bull,44.0
14,ANT,Mercedes,18.5
18,RUS,Mercedes,8.2
10,HAM,Ferrari,6.5
16,HAD,Red Bull,3.6
6,NOR,McLaren,1.8
15,LEC,Ferrari,1.3
0,PIA,McLaren,1.3
5,GAS,Alpine F1 Team,1.1
2,BOR,Audi,1.1
